# Ch15 (A, inline) - Evaluating the RAG with a Designed Experiment

The DoE cohort (Ch6) is the test set, the GMS is the oracle. We measure precision/recall@k, calibrated correctness, completeness, and attribute failures to the presentation factors.

In [ ]:
# knowlytix and forgeloop are installed from PyPI (pip install knowlytix forgeloop)
import os, sys
REPO = os.path.join(os.path.dirname(os.getcwd()), "code") if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()
sys.path.insert(0, os.path.join(REPO, "scripts"))

In [ ]:
import capstone_pipeline as cp
store = cp.load_store(os.path.join(REPO, "data", "gms_annual_report_store"))
pipe = cp.build_pipeline(store, cp.make_qwen(), accept_threshold=0.0)

In [ ]:
import json, re
from knowlytix.harness.testing.hallucination import HallucinationOracle
from knowlytix.harness.testing.completeness import CompletenessEvaluator
oracle, comp = HallucinationOracle(store=store), CompletenessEvaluator(store)
cohort = json.load(open(os.path.join(REPO,"data","enrichment","rag_cohort.json")))
def gtv(e): return str(e[1]) if isinstance(e, list) else str(e)
def golden(c):
    g = gtv(c["expected_answer"])
    return [(h,r,str(t)) for h,r,t in store.triples if str(t)==g][:1]

In [ ]:
P=R=C=CO=AB=N=NA=0
for c in cohort:
    a = pipe.query(c["question"]); g = gtv(c["expected_answer"]); N+=1
    top = a.sources[:3]; hit = [f for f in top if str(f.tail)==g]
    P += len(hit)/max(1,len(top)); R += 1.0 if hit else 0.0
    if a.decision != "accept": AB += 1; continue
    NA += 1; gold = golden(c)
    if gold:
        h,r,_ = gold[0]; m = re.findall(r"\d[\d,]*\.?\d*", (a.answer or "").replace(",",""))
        CO += 1.0 if (m and oracle.assess_claim(h,r,m[0]).passed) else 0.0
        C += comp.evaluate(a.answer or "", [g], {"head": h}).score
print(f"precision@3={P/N:.3f} recall@3={R/N:.3f} correctness={CO/max(1,NA):.3f} "
      f"completeness={C/max(1,NA):.3f} abstention={AB/N:.3f}")